### Step 1: Import Libraries

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [15]:
import re
import string
import collections

In [4]:
import nltk
from nltk.stem import WordNetLemmatizer

### Step 2: Constant

In [30]:
FILE_PATH = './data/AutocorrectorFeature.txt'
LETTERS = string.ascii_lowercase

### Step 3: Load Data

In [8]:
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    text_data = f.read().lower()
    words = re.findall(r'\w+', text_data)

In [12]:
vocabs = set(words)

### Step 4: Exploaring Data

#### Count Word Frequency

In [17]:
word_count = collections.Counter(words)

#### Calculate Word Probability

In [22]:
probabilities = {word: count/len(words) for word, count in word_count.items()}

### Step 5: Define NLP-Based Functions

In [24]:
lemmatizer = WordNetLemmatizer()

In [25]:
def lemmatize_word(word):
    return lemmatizer.lemmatize(word)

In [26]:
def delete_letter(word):
    return [word[:i] + word[i+1:] for i in range(len(word))]

In [28]:
def swap_letters(word):
    return [word[:i] + word[i+1] + word[i] + word[i+2:] for i in range(len(word)-1)]

In [31]:
def replace_letter(word):
    return [word[:i] + l + word[i+1:] for i in range(len(word)) for l in LETTERS]

In [32]:
def insert_letter(word):
    return [word[:i] + l + word[i:] for i in range(len(word)+1) for l in LETTERS]

### Step 6: Generate Candidate Corrections

In [33]:
def generate_candidates(word):
    candidates = set()
    candidates.update(delete_letter(word))
    candidates.update(swap_letters(word))
    candidates.update(replace_letter(word))
    candidates.update(insert_letter(word))
    return candidates

In [34]:
def generate_candidates_level2(word):
    level1 = generate_candidates(word)
    level2 = set()
    for w in level1:
        level2.update(generate_candidates(w))
    return level2

### Step 7: Get the Best Corrections

In [35]:
def get_best_correction(word, probs, vocab, max_suggestions=3):
    if word in vocab:
        return [(word, probs.get(word, 0))]
    candidates = generate_candidates(word).intersection(vocab)
    if not candidates:
        candidates = generate_candidates_level2(word).intersection(vocab)
    if not candidates:
        return list()
    scored = [(w, probs.get(w, 1e-8)) for w in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:max_suggestions]

### Step 8: User Input & Output Suggestions

In [41]:
user_input = input("Enter a word for autocorrection: ")
suggestions = get_best_correction(user_input, probabilities, vocabs, max_suggestions=3)
print("\nTop suggestions:")
for suggestion in suggestions:
    print(suggestion[0])

Enter a word for autocorrection:  daedd



Top suggestions:
added
dead
died
